# Du or Sie? Formality in English→German Machine Translation

**Research questions**
1. **RQ1 – Default:** Which form of address (*Sie* = formal, *du* = informal) do MT systems choose when the English source gives no explicit instruction?
2. **RQ2 – Context:** Does that choice change with the conversational setting (customer service vs. phone conversation vs. casual chat)?
3. **RQ3 – Control:** Can an instruction in the prompt steer LLM translations to the requested form, and what does that cost in translation quality?

**Data:** CoCoA-MT EN→DE test set (Nadejde et al., 2022): 600 source segments, 200 per domain, each with a formal and an informal reference translation.

**Systems:** Opus-MT (bilingual NMT), NLLB-200 600M (multilingual NMT), Qwen2.5-Instruct 1.5B and 3B (LLMs, three prompt conditions each).

---
### How to run
1. `Runtime → Change runtime type → T4 GPU` → Save
2. `Runtime → Run all`
3. Wait (≈ 30–45 min). At the end, **`outputs.zip`** downloads automatically.

If Colab disconnects, just run all again: finished translations are cached in `outputs/hyp/` and skipped.

## 1. Setup

In [ ]:
!pip -q install "transformers==4.51.3" accelerate sentencepiece "sacrebleu>=2.3"
![ -d contrastive-controlled-mt ] || git clone -q https://github.com/amazon-science/contrastive-controlled-mt.git
!cd contrastive-controlled-mt && git checkout -q 441e23a7c41beeac6329ffdb27d47024eb71b829 && git log -1 --format="dataset commit: %H"

In [ ]:
import os, re, gc, sys, json, time, random, zipfile, platform
import numpy as np
import pandas as pd
import torch

# ---------------- configuration ----------------
QUICK_TEST = False   # True = only 10 segments per domain (pipeline check, ~5 min)

DATA_DIR = "contrastive-controlled-mt/CoCoA-MT/test/en-de"
SCORER_DIR = "contrastive-controlled-mt/IWSLT2022"
OUT_DIR = "outputs"
HYP_DIR = f"{OUT_DIR}/hyp"
FIG_DIR = f"{OUT_DIR}/figures"
DOMAINS = ["call_center", "telephony", "topical_chat"]
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NMT_SYSTEMS = {                      # classic neural MT: no formality control possible
    "opus-mt": "Helsinki-NLP/opus-mt-en-de",
    "nllb-600m": "facebook/nllb-200-distilled-600M",
}
LLM_SYSTEMS = {                      # instruction-tuned LLMs: three prompt conditions
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
}
SYSTEM_PROMPT = "You are a professional English-to-German translator."
PROMPTS = {
    "neutral":  "Translate the following English text into German. "
                "Reply with the German translation only.\n\nEnglish: {src}",
    "formal":   "Translate the following English text into German. "
                "Use the formal form of address (\"Sie\"). "
                "Reply with the German translation only.\n\nEnglish: {src}",
    "informal": "Translate the following English text into German. "
                "Use the informal form of address (\"du\"). "
                "Reply with the German translation only.\n\nEnglish: {src}",
}
NMT_GEN = dict(num_beams=4, max_new_tokens=256)
LLM_GEN = dict(do_sample=False, max_new_tokens=256, repetition_penalty=1.05)  # greedy decoding

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
for d in (OUT_DIR, HYP_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if DEVICE == "cpu":
    print("WARNING: no GPU. Set Runtime → Change runtime type → T4 GPU, otherwise the LLM part is very slow.")

## 2. Load CoCoA-MT (EN→DE test set)

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n").strip() for line in f]

def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line.replace("\n", " ").strip() + "\n")

rows = []
for dom in DOMAINS:
    base = f"{DATA_DIR}/formality-control.test.{dom}.en-de"
    src = read_lines(base)
    refs = {k: read_lines(f"{base}.{k}") for k in
            ["formal", "informal", "formal.annotated", "informal.annotated"]}
    assert all(len(v) == len(src) for v in refs.values()), dom
    for i, s in enumerate(src):
        rows.append(dict(domain=dom, idx=i, src=s,
                         ref_formal=refs["formal"][i], ref_informal=refs["informal"][i],
                         ann_formal=refs["formal.annotated"][i], ann_informal=refs["informal.annotated"][i]))
data = pd.DataFrame(rows)
if QUICK_TEST:
    data = data.groupby("domain").head(10).reset_index(drop=True)
SRC = data["src"].tolist()
print(data.groupby("domain").size().to_string())
data[["domain", "src", "ref_formal", "ref_informal"]].head(3)

## 3. Translation functions

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

def length_sorted_batches(texts, batch_size):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for start in range(0, len(order), batch_size):
        yield order[start:start + batch_size]

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def translate_nmt(model_name, sents, batch_size=32):
    is_nllb = "nllb" in model_name
    tok = AutoTokenizer.from_pretrained(model_name, **({"src_lang": "eng_Latn"} if is_nllb else {}))
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()   # fp32: small models
    gen_kw = dict(NMT_GEN)
    if is_nllb:
        gen_kw["forced_bos_token_id"] = tok.convert_tokens_to_ids("deu_Latn")
    out = [None] * len(sents)
    for b in length_sorted_batches(sents, batch_size):
        enc = tok([sents[i] for i in b], return_tensors="pt", padding=True,
                  truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            gen = model.generate(**enc, **gen_kw)
        for i, g in zip(b, gen):
            out[i] = tok.decode(g, skip_special_tokens=True).strip()
    free_gpu(model)
    return out

def clean_llm_output(text):
    t = text.strip()
    t = re.split(r"\n\s*\n", t)[0]                       # keep the first paragraph only
    t = re.sub(r"^(German|Deutsch|Translation|Übersetzung)\s*:\s*", "", t, flags=re.I)
    t = t.replace("\n", " ").strip()
    if len(t) >= 2 and t[0] in "\"„“'" and t[-1] in "\"“”'":
        t = t[1:-1].strip()                              # strip wrapping quotes
    return t

def load_llm(model_name, dtype):
    tok = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype,
                                                 device_map=DEVICE).eval()
    return tok, model

def translate_llm(tok, model, sents, template, batch_size=16):
    prompts = [tok.apply_chat_template(
                   [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": template.format(src=s)}],
                   tokenize=False, add_generation_prompt=True) for s in sents]
    out, raw = [None] * len(sents), [None] * len(sents)
    for b in length_sorted_batches(prompts, batch_size):
        enc = tok([prompts[i] for i in b], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            gen = model.generate(**enc, **LLM_GEN, pad_token_id=tok.pad_token_id)
        dec = [tok.decode(g, skip_special_tokens=True) for g in gen[:, enc["input_ids"].shape[1]:]]
        for i, t in zip(b, dec):
            raw[i], out[i] = t, clean_llm_output(t)
    return out, raw

def looks_broken(outputs):
    # fp16 overflow in some models yields empty strings or "!!!!" – detect and fall back to fp32
    return sum(len(re.findall(r"[A-Za-zÄÖÜäöüß]", o)) >= 3 for o in outputs) < len(outputs) / 2

## 4. Run all systems (cached)

In [ ]:
def hyp_path(system, cond):
    return f"{HYP_DIR}/{system}__{cond}.de"

def is_done(path):
    return os.path.exists(path) and len(read_lines(path)) == len(SRC)

timings = {}

# --- classic NMT (single, uncontrolled condition) ---
for name, model_id in NMT_SYSTEMS.items():
    p = hyp_path(name, "neutral")
    if is_done(p):
        print(f"[skip] {name}"); continue
    t0 = time.time()
    hyps = translate_nmt(model_id, SRC)
    write_lines(p, hyps)
    timings[f"{name}__neutral"] = round(time.time() - t0, 1)
    print(f"[done] {name}: {timings[f'{name}__neutral']} s | e.g. {hyps[0]}")

# --- LLMs (neutral / formal / informal prompts) ---
for name, model_id in LLM_SYSTEMS.items():
    todo = [c for c in PROMPTS if not is_done(hyp_path(name, c))]
    if not todo:
        print(f"[skip] {name}"); continue
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    tok, model = load_llm(model_id, dtype)
    probe, _ = translate_llm(tok, model, SRC[:4], PROMPTS["neutral"])
    if looks_broken(probe):
        print(f"{name}: fp16 output looks broken {probe[:2]} -> reloading in fp32")
        free_gpu(model)
        dtype = torch.float32
        tok, model = load_llm(model_id, dtype)
    for cond in todo:
        t0 = time.time()
        hyps, raw = translate_llm(tok, model, SRC, PROMPTS[cond])
        write_lines(hyp_path(name, cond), hyps)
        with open(f"{HYP_DIR}/{name}__{cond}.raw.jsonl", "w", encoding="utf-8") as f:
            for r in raw:
                f.write(json.dumps({"raw": r}, ensure_ascii=False) + "\n")
        timings[f"{name}__{cond}"] = round(time.time() - t0, 1)
        print(f"[done] {name} / {cond} ({dtype}): {timings[f'{name}__{cond}']} s | e.g. {hyps[0]}")
    free_gpu(model)

## 5. Scoring

* **Formality label (primary):** the official IWSLT 2022 / CoCoA-MT phrase-matching scorer. A translation is `FORMAL` if it contains annotated formality phrases of the formal reference and none of the informal one, `INFORMAL` vice versa, `NEUTRAL` if it contains neither, `OTHER` if both. Punctuation is stripped before matching (the original whitespace split misses e.g. *„haben Sie?“*); the unmodified version is kept as `label_strict`.
* **Formality label (robustness check):** a reference-free lexical detector (informal: *du, dich, dir, dein-, euch, euer-*; formal: capitalised *Sie, Ihnen, Ihr-* not at the start of a sentence).
* **Translation quality:** chrF and BLEU (sacreBLEU). Uncontrolled output is scored against both references (multi-reference); controlled output against the reference with the requested formality.

In [ ]:
import sacrebleu
sys.path.insert(0, SCORER_DIR)
import scorer as official   # contrastive-controlled-mt/IWSLT2022/scorer.py (Amazon, MIT-0)

def words(text):
    return re.findall(r"\w+", text)

def official_label(hyp, ann_formal, ann_informal):
    # Same phrase-matching logic as official.get_matching_phrases (all tokens of an annotated
    # phrase must occur in the hypothesis), but punctuation is stripped first: the original
    # whitespace split misses phrases followed by punctuation, e.g. "haben Sie?".
    hyp_words = set(words(hyp))
    def matched(ann):
        return [ph for _, ph in official.FORMALITY_PHRASES.findall(ann) if set(words(ph)) <= hyp_words]
    return official.predict_formality_label(matched(ann_formal), matched(ann_informal))

def official_label_strict(hyp, ann_formal, ann_informal):
    # unmodified official scorer logic, reported for comparability
    f = official.get_matching_phrases(hyp, ann_formal, True, official.FORMALITY_PHRASES)
    i = official.get_matching_phrases(hyp, ann_informal, True, official.FORMALITY_PHRASES)
    return official.predict_formality_label(f, i)

INFORMAL_WORDS = {"du", "dich", "dir", "dein", "deine", "deinen", "deinem", "deiner", "deines",
                  "euch", "euer", "eure", "euren", "eurem", "eurer", "eures"}
FORMAL_WORDS = {"Sie", "Ihnen", "Ihr", "Ihre", "Ihren", "Ihrem", "Ihrer", "Ihres"}

def lexical_label(text):
    n_inf = n_form = 0
    sentence_start = True
    for tok in re.findall(r"\w+|[.!?:]", text):
        if tok in ".!?:":
            sentence_start = True
            continue
        if tok.lower() in INFORMAL_WORDS:
            n_inf += 1
        elif tok in FORMAL_WORDS and not sentence_start:   # sentence-initial "Sie" may mean she/they
            n_form += 1
        sentence_start = False
    if n_form and not n_inf: return "FORMAL"
    if n_inf and not n_form: return "INFORMAL"
    if not n_inf and not n_form: return "NEUTRAL"
    return "OTHER"

# sanity check: the detectors must recover the reference labels
for col, expected in [("ref_formal", "FORMAL"), ("ref_informal", "INFORMAL")]:
    triples = list(zip(data[col], data["ann_formal"], data["ann_informal"]))
    strict = np.mean([official_label_strict(*t) == expected for t in triples])
    off = np.mean([official_label(*t) == expected for t in triples])
    lex = np.mean([lexical_label(h) == expected for h in data[col]])
    print(f"{col} labelled {expected}: official strict {strict:.1%} | "
          f"official, punctuation-robust {off:.1%} | lexical detector {lex:.1%}")

In [ ]:
RUNS = [(s, "neutral") for s in NMT_SYSTEMS] + [(s, c) for s in LLM_SYSTEMS for c in PROMPTS]
RUNS = [(s, c) for s, c in RUNS if os.path.exists(hyp_path(s, c))]

per_sentence = []
for system, cond in RUNS:
    hyps = read_lines(hyp_path(system, cond))
    for (_, r), h in zip(data.iterrows(), hyps):
        per_sentence.append(dict(
            system=system, family="NMT" if system in NMT_SYSTEMS else "LLM", condition=cond,
            domain=r.domain, idx=r.idx, src=r.src, hyp=h,
            label=official_label(h, r.ann_formal, r.ann_informal),
            label_strict=official_label_strict(h, r.ann_formal, r.ann_informal),
            label_lexical=lexical_label(h)))
per = pd.DataFrame(per_sentence)
per.to_csv(f"{OUT_DIR}/per_sentence.csv", index=False)

LABELS = ["FORMAL", "INFORMAL", "NEUTRAL", "OTHER"]

def label_table(df, col="label"):
    frames = []
    for keys, g in df.groupby(["system", "condition", "domain"]):
        frames.append(dict(zip(["system", "condition", "domain"], keys),
                           **{L: int((g[col] == L).sum()) for L in LABELS}, n=len(g)))
    t = pd.DataFrame(frames)
    tot = t.groupby(["system", "condition"])[LABELS + ["n"]].sum().reset_index()
    tot["domain"] = "ALL"
    t = pd.concat([t, tot], ignore_index=True)
    matched = t["FORMAL"] + t["INFORMAL"]
    t["pct_formal_of_matched"] = (100 * t["FORMAL"] / matched.replace(0, np.nan)).round(1)
    t["pct_informal_of_matched"] = (100 * t["INFORMAL"] / matched.replace(0, np.nan)).round(1)
    t["pct_neutral"] = (100 * t["NEUTRAL"] / t["n"]).round(1)
    return t

labels_official = label_table(per, "label")
labels_strict = label_table(per, "label_strict")
labels_lexical = label_table(per, "label_lexical")
labels_official.to_csv(f"{OUT_DIR}/formality_official.csv", index=False)
labels_strict.to_csv(f"{OUT_DIR}/formality_official_strict.csv", index=False)
labels_lexical.to_csv(f"{OUT_DIR}/formality_lexical.csv", index=False)
labels_official[labels_official.domain == "ALL"]

In [ ]:
# translation quality (corpus-level chrF / BLEU)
quality = []
refs_by_key = data.set_index(["domain", "idx"])
for system, cond in RUNS:
    sub = per[(per.system == system) & (per.condition == cond)]
    for dom in DOMAINS + ["ALL"]:
        s = sub if dom == "ALL" else sub[sub.domain == dom]
        d = refs_by_key.loc[list(zip(s.domain, s.idx))]
        hyps = s.hyp.tolist()
        if cond == "formal":
            refs = [d.ref_formal.tolist()]
        elif cond == "informal":
            refs = [d.ref_informal.tolist()]
        else:
            refs = [d.ref_formal.tolist(), d.ref_informal.tolist()]
        quality.append(dict(system=system, condition=cond, domain=dom,
                            chrF=round(sacrebleu.corpus_chrf(hyps, refs).score, 1),
                            BLEU=round(sacrebleu.corpus_bleu(hyps, refs).score, 1)))
quality = pd.DataFrame(quality)
quality.to_csv(f"{OUT_DIR}/quality.csv", index=False)
quality[quality.domain == "ALL"]

## 6. Statistical tests

In [ ]:
from scipy.stats import chi2_contingency, binomtest

tests = []
for system, cond in RUNS:
    sub = per[(per.system == system) & (per.condition == cond) & per.label.isin(["FORMAL", "INFORMAL"])]
    n_f, n = int((sub.label == "FORMAL").sum()), len(sub)
    row = dict(system=system, condition=cond, n_matched=n, n_formal=n_f,
               pct_formal=round(100 * n_f / n, 1) if n else np.nan)
    # RQ1: is the default different from 50/50?
    if n:
        row["binom_p_vs_50pct"] = binomtest(n_f, n, 0.5).pvalue
    # RQ2: does the formal/informal choice depend on the domain? (chi-square test of independence)
    ct = pd.crosstab(sub.domain, sub.label).reindex(columns=["FORMAL", "INFORMAL"], fill_value=0)
    if (ct.sum(axis=0) > 0).all() and (ct.sum(axis=1) > 0).sum() > 1:
        chi2, p, dof, _ = chi2_contingency(ct)
        row.update(chi2=round(chi2, 2), dof=dof, chi2_p=p,
                   cramers_v=round(np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), 3))
    else:
        row.update(chi2=np.nan, dof=np.nan, chi2_p=np.nan, cramers_v=np.nan)  # no variation
    tests.append(row)
tests = pd.DataFrame(tests)
tests.to_csv(f"{OUT_DIR}/stat_tests.csv", index=False)
tests

## 7. Quick-look figures

In [ ]:
import matplotlib.pyplot as plt

neutral = labels_official[(labels_official.condition == "neutral") & (labels_official.domain != "ALL")]
fig, ax = plt.subplots(figsize=(9, 4.5))
systems = list(NMT_SYSTEMS) + list(LLM_SYSTEMS)
width = 0.8 / len(systems)
for k, s in enumerate(systems):
    v = neutral[neutral.system == s].set_index("domain").reindex(DOMAINS)["pct_formal_of_matched"]
    ax.bar(np.arange(len(DOMAINS)) + k * width, v.values, width, label=s)
ax.axhline(50, color="grey", lw=1, ls="--")
ax.set_xticks(np.arange(len(DOMAINS)) + width * (len(systems) - 1) / 2, DOMAINS)
ax.set_ylabel("% formal (Sie) among matched")
ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=8)
ax.set_title("Uncontrolled translation: share of formal address by domain")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/quicklook_default_formality.png", dpi=300)

ctrl = labels_official[(labels_official.domain == "ALL") & (labels_official.system.isin(LLM_SYSTEMS))]
print(ctrl[["system", "condition", "pct_formal_of_matched", "pct_informal_of_matched", "pct_neutral"]].to_string(index=False))

## 8. Save and download results

In [ ]:
import transformers
env = dict(date=time.strftime("%Y-%m-%d %H:%M"), device=DEVICE,
           gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
           python=platform.python_version(), torch=torch.__version__,
           transformers=transformers.__version__, sacrebleu=sacrebleu.__version__,
           dataset_commit="441e23a7c41beeac6329ffdb27d47024eb71b829",
           quick_test=QUICK_TEST, n_segments=len(SRC),
           nmt_systems=NMT_SYSTEMS, llm_systems=LLM_SYSTEMS,
           system_prompt=SYSTEM_PROMPT, prompts=PROMPTS,
           nmt_generation=NMT_GEN, llm_generation=LLM_GEN, timings_sec=timings)
with open(f"{OUT_DIR}/env.json", "w") as f:
    json.dump(env, f, indent=2, ensure_ascii=False)

with zipfile.ZipFile("outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            z.write(os.path.join(root, fn))
print("outputs.zip:", round(os.path.getsize("outputs.zip") / 1e6, 2), "MB")
try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not running in Colab – outputs.zip is in the working directory.")